In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q04/q04_rewrite/checkpoints/pre_cell_2.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Extract unique L_ORDERKEY values where commit date is before receipt date
fline_keys = (
    lineitem
    .loc[lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE, ['L_ORDERKEY']]
    .drop_duplicates()
    .rename(columns={'L_ORDERKEY': 'O_ORDERKEY'})
)

# Filter orders by date range, join to filtered lineitem keys, and count orders per priority
total = (
    orders
    .loc[
        (orders.O_ORDERDATE >= '1993-08-01') &
        (orders.O_ORDERDATE < '1993-11-01')
    ]
    .merge(fline_keys, on='O_ORDERKEY', how='inner')
    .groupby('O_ORDERPRIORITY')
    .size()
    .reset_index(name='O_ORDERKEY')
)

CPU times: user 63.4 ms, sys: 35.8 ms, total: 99.2 ms
Wall time: 107 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_small_q04/checkpoints/post_cell_2_try_2.pickle